# **Подгоовка** данных по территории

## 1. Получение кварталов

### 1.1. Получение данных из **OpenStreetMap**. 
Этот шаг можно пропустить, если данные уже существуют.

In [ ]:
import osmnx as ox

boundaries = ox.geocode_to_gdf('Северодонецк, Украина')
boundaries.plot().set_axis_off()

In [ ]:
bc_tags = {
    'roads': {
      "highway": ["construction","crossing","living_street","motorway","motorway_link","motorway_junction","pedestrian","primary","primary_link","raceway","residential","road","secondary","secondary_link","services","tertiary","tertiary_link","track","trunk","trunk_link","turning_circle","turning_loop","unclassified",],
      "service": ["living_street", "emergency_access"]
    },
    'railways': {
      "railway": "rail"
    },
    'water': {
      'riverbank':True,
      'reservoir':True,
      'basin':True,
      'dock':True,
      'canal':True,
      'pond':True,
      'natural':['water','bay'],
      'waterway':['river','canal','ditch'],
      'landuse':'basin',
      'water': 'lake'
    }
}

In [ ]:
water = ox.features_from_polygon(boundaries.union_all(), bc_tags['water'])
roads = ox.features_from_polygon(boundaries.union_all(), bc_tags['roads'])
railways = ox.features_from_polygon(boundaries.union_all(), bc_tags['railways'])

In [ ]:
water = water[water.geom_type.isin(['Polygon', 'MultiPolygon', 'LineString', 'MultiLineString'])].copy()
roads = roads[roads.geom_type.isin(['LineString', 'MultiLineString'])].copy()
railways = railways[railways.geom_type.isin(['LineString', 'MultiLineString'])].copy()

In [ ]:
crs = boundaries.estimate_utm_crs()

for gdf in [water, roads, railways, boundaries]:
  gdf.to_crs(crs, inplace=True)

### 1.2. Предварительная обработка входной геометрии

Этот шаг можно пропустить, если входная геометрия уже отсортирована как линии, многоугольники и границы.

In [ ]:
roads = roads.reset_index(drop=True)
railways = railways.reset_index(drop=True)
water = water.reset_index(drop=True)

In [ ]:
from blocksnet.blocks.cutting import preprocess_urban_objects, cut_urban_blocks

lines, polygons = preprocess_urban_objects(roads, railways, water)

### 1.3. **Нарезка** городских кварталов

In [ ]:
blocks = cut_urban_blocks(boundaries, lines, polygons)
blocks.plot().set_axis_off()

___


## 2. Назначение **землепользования**


### 2.1. Получение данных из OpenStreetMap

Этот шаг можно пропустить, если слой функциональных зон уже получен из государственных источников.

In [ ]:
import osmnx as ox

functional_zones = ox.features_from_polygon(boundaries.to_crs(4326).union_all(), tags={'landuse': True})

In [ ]:
functional_zones = functional_zones.reset_index(drop=True)[['geometry','landuse']].rename(columns={'landuse': 'functional_zone'})
functional_zones = functional_zones.to_crs(crs)
functional_zones.head()


### 2.2. Указание правил

Правила определяют, как столбец `functional_zone` будет отображаться в `LandUse`.

In [ ]:
from blocksnet.enums import LandUse

rules = {
  'commercial': LandUse.BUSINESS,
  'industrial': LandUse.INDUSTRIAL,
  'cemetery': LandUse.SPECIAL,
  'garages': LandUse.INDUSTRIAL,
  'residential': LandUse.RESIDENTIAL,
  'retail': LandUse.BUSINESS,
  'grass': LandUse.RECREATION,
  'farmland': LandUse.AGRICULTURE,
  'construction': LandUse.SPECIAL,
  'brownfield': LandUse.INDUSTRIAL,
  'forest': LandUse.RECREATION,
  'recreation_ground': LandUse.RECREATION,
  'religious': LandUse.SPECIAL,
  'flowerbed': LandUse.RECREATION,
  'military': LandUse.SPECIAL,
  'landfill': LandUse.TRANSPORT
}

### 2.3. Назначение земплепользования 

In [ ]:
from blocksnet.blocks.assignment import assign_land_use

blocks = assign_land_use(blocks, functional_zones.reset_index(drop=True).rename(columns={'landuse': 'functional_zone'}), rules)

In [ ]:
blocks.head()

In [ ]:
ax = blocks.plot(color='#ddd')
blocks.plot(column='land_use', legend=True, ax=ax).set_axis_off()

___

## 3. Расчет матрицы доступности

Существует несколько способов определения пространственных связей между городскими кварталами:
- **Матрица доступности** - матрица `n ^ 2`, которая определяет время в пути между центроидами городских кварталов в минутах.
- **Матрица расстояний** - матрица `n ^ 2`, которая определяет евклидово расстояние между центроидами городских кварталов в метрах.
- **График смежности** - график с `n` узлами. Каждое ребро этого графика указывает на пространственную смежность двух городских кварталов.

В этом примере **матрица доступности** будет рассчитана с использованием библиотеки IduEdu.

### 3.1. Построение интермодальных графов

Существует несколько типов городских графов для моделирования городской сети:
- **walk** - пешеходная сеть.
- **drive** - сеть личного транспорта.
- **интермодальная** - сеть пешеходного и общественного транспорта.

В этом примере будет построен **интермодальный** график, но в целом это зависит от результатов исследования.

In [ ]:
from blocksnet.relations import get_accessibility_graph

graph = get_accessibility_graph(boundaries, 'intermodal')

### 3.2. Вычисление матрицы доступности

In [ ]:
from blocksnet.relations import calculate_accessibility_matrix

acc_mx = calculate_accessibility_matrix(blocks, graph)

In [ ]:
acc_mx.head()

### 3.3 Вычисления **связанности**

#### Вычисления **площади участка**

In [ ]:
import pandas as pd

blocks['site_area'] = blocks.area
blocks.head()

In [ ]:
from blocksnet.analysis.network.accessibility import area_accessibility
# matrix = pd.read_pickle('./data/niipg/matrix.pickle')

In [ ]:
area_acc = area_accessibility(acc_mx, blocks)
blocks = blocks.join(area_acc)

In [ ]:
blocks.plot('area_accessibility', legend=True).set_axis_off()

___

## 4. Вычисления **показателей застройки**

Здания определяют параметры городской среды в разных блоках. Для некоторых методов blocksnet может использоваться только население, но в целом здание определяется с помощью:

- `population : float`
- `footprint_area : float` - базовая площадь здания.
- `number_of_floors : float`
- `build_floor_area : float` - суммарная площадь каждого этажа здания
- `is_living : bool`
- `жилая_площадь : float` - площадь, определенная для жителей
- `non_living_area : float` - область, в которой не определено место проживания

Некоторые из этих параметров могут быть отсутствовать в данных, и в этом случае они будут установлены в значение по умолчанию.

### 4.1. Извлечение зданий из OpenStreetMap

Этот шаг можно пропустить, если данные уже получены из любого доступного источника.

In [ ]:
buildings = ox.features_from_polygon(boundaries.to_crs(4326).union_all(), tags={'building': True}).reset_index(drop=True).to_crs(crs)

#### 4.1.1. `is_living`

In [ ]:
is_living_tags = ['residential', 'house', 'apartments', 'detached', 'terrace', 'dormitory']
buildings['is_living'] = buildings['building'].apply(lambda b : b in is_living_tags)

#### 4.1.2. `number_of_floors`

In [ ]:
import pandas as pd

buildings['number_of_floors'] = pd.to_numeric(buildings['building:levels'], errors='coerce')
buildings['number_of_floors'] = buildings['number_of_floors'].clip(lower=1)

In [ ]:
required = {'site_area', 'footprint_area', 'build_floor_area', 'living_area'}
missing = required - set(buildings.columns)
print(missing)

### 4.2. **Восполнение** недостающих данных

In [ ]:
from blocksnet.preprocessing.imputing import impute_buildings

buildings = impute_buildings(buildings, default_living_demand=30)

In [ ]:
buildings.sample(5)

In [ ]:
buildings.population.sum()

### 4.3. Перенос зданий в городские кварталы

In [ ]:
from blocksnet.blocks.aggregation import aggregate_objects

buildings_blocks = aggregate_objects(blocks, buildings)[0]

In [ ]:
buildings_blocks.head()

In [ ]:
buildings_blocks.plot('population', legend=True).set_axis_off()

In [ ]:
blocks = blocks.join(buildings_blocks.drop(columns=['geometry']))
blocks.head()

_____


## 5. Морфотипы 

### 5.1 Вычисление плотности застройки

In [ ]:
from blocksnet.analysis.indicators import calculate_density_indicators 

density_df = calculate_density_indicators(blocks[[
    'site_area',
    'footprint_area',
    'build_floor_area',
    'living_area'
]])
density_df.head()

In [ ]:
blocks.loc[:, density_df.columns] = density_df
blocks.head()

In [ ]:
blocks.plot('non_living_area', legend=True).set_axis_off()

### 5.1 Вычисления морфотипов

In [ ]:
# from blocksnet.analysis.morphotypes import get_spacematrix_morphotypes

# spacematrix_df, spacematrix_clusters = get_spacematrix_morphotypes(blocks)
# blocks.loc[:, spacematrix_df.columns] = spacematrix_df
# blocks.head()

In [ ]:
from blocksnet.analysis.morphotypes import get_strelka_morphotypes

strelka_df = get_strelka_morphotypes(blocks)
blocks.loc[:, strelka_df.columns] = strelka_df
blocks.head()

In [ ]:
# blocks.to_file('severodonetsk_blocks.geojson')

# **Urbanomy**

In [ ]:
import geopandas as gpd
import numpy as np
blocks = gpd.read_file('./data/severodonetsk_blocks.geojson')

In [ ]:
cols = [
    'residential','business','recreation','industrial','transport','special',
    'agriculture','land_use','share','footprint_area','build_floor_area',
    'living_area','non_living_area','population','site_area','fsi','gsi',
    'mxi','l','morphotype','area_accessibility', 'geometry'
]
basline_blocks = blocks[cols]
basline_blocks.head()

___

## Предсказание **стоимости земельных участков**

In [ ]:
from catboost import CatBoostRegressor

from urbanomy.methods.land_value_modeling import LandPriceEstimator

model = CatBoostRegressor()
model.load_model('./data/catboost_model.cbm')  # модель на лог-цене

estimator = LandPriceEstimator(
    model=model,
    blocks=basline_blocks,
)
blocks_pred = estimator.predict()
blocks_pred.head()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
# 1) Цена за сотку (100 м²)
blocks_pred["land_value_per_100m2"] = blocks_pred["land_value"] / blocks_pred["site_area"] * 100

# 2) Заменяем inf на NaN
blocks_pred = blocks_pred.replace([np.inf, -np.inf], np.nan)
blocks_pred = blocks_pred.fillna(0)

# 3) Удаляем выбросы: оставляем данные до 99-го перцентиля
p99 = blocks_pred["land_value_per_100m2"].quantile(0.99)
blocks_clean = blocks_pred[blocks_pred["land_value_per_100m2"] <= p99].copy()

print(blocks_clean["land_value_per_100m2"].describe())

# 4) Гистограмма после очистки
plt.figure()
blocks_clean["land_value_per_100m2"].dropna().hist(bins=50)
plt.xlabel("Цена за сотку (руб.)")
plt.ylabel("Частота")
plt.title("Распределение цены за сотку")
plt.show()

# 5) Box-plot после очистки
plt.figure()
plt.boxplot(blocks_clean["land_value_per_100m2"].dropna(), vert=False)
plt.xlabel("Цена за сотку (руб.)")
plt.title("Box-plot цены за сотку")
plt.show()


In [ ]:
import matplotlib.pyplot as plt

blocks_clean.plot(
    column='land_value_per_100m2',
    legend=True,
    figsize=(15,15),
    cmap='coolwarm'
).set_axis_off()
plt.title('Цена земли за сотку (руб.)', fontsize=16)

plt.show()

In [ ]:
import matplotlib.pyplot as plt

blocks_clean.plot(
    column='land_value',
    legend=True,
    figsize=(15,15),
    cmap='coolwarm'
).set_axis_off()
plt.title('Цены земли (руб.)', fontsize=16)

plt.show()

## Сценарное изменение земельных участков

In [ ]:
blocks_clean["id"] = blocks_clean.index


In [ ]:
import matplotlib.pyplot as plt
basline_blocks["id"] = basline_blocks.index

fig, ax = plt.subplots(figsize=(25, 35))
basline_blocks.plot(ax=ax, color="lightgrey", edgecolor="white", linewidth=0.6)  # фон
basline_blocks.loc[basline_blocks["id"]==147].plot(
    ax=ax, color="none", edgecolor="gold", linewidth=5.5
)
ax.set_title("Изменяемый квартал в Северодонецке", fontsize=20)
ax.axis("off")
plt.show()

In [ ]:
blocks_clean.loc[142]

In [ ]:
from urbanomy.methods.land_value_modeling import (
    ScenarioTEPModifier,
    plot_scenario_impact,
    transfer_baseline_prices,
)

blocks_before = basline_blocks.copy()

changes = {
    # Структура использования территории
    "land_use": "LandUse.RESIDENTIAL",
    "residential": 1.0,
    "business": 0.0,
    "recreation": 0.0,
    "industrial": 0.0,
    "transport": 0.0,
    "special": 0.0,
    "agriculture": 0.0,
    "share": 1.0,

    "footprint_area": 48170.0451624,    # ≈ 60% от site_area
    "build_floor_area": 481700.451624,  # FSI ≈ 6.0
    "living_area": 337190.3161368,      # ≈ 70% 
    "non_living_area": 144510.1354872,  # ≈ 30% 
    "population": 5537,                # 


    "fsi": 6.0,                         # build_floor_area / site_area
    "gsi": 0.60,                        # footprint_area / site_area
    "l": 10.0,                          # build_floor_area / footprint_area (этажность)
                    # build_floor_area / footprint_area (этажность)

    # Морфотип – высотный дом
    "morphotype": "high-rise residential",
    }

target_idx = 143

modifier = ScenarioTEPModifier(blocks_before)
blocks_after = modifier.apply(target_idx, changes)

# 1) Предсказание стоимости до/после
before_estimator = LandPriceEstimator(model=model, blocks=blocks_before)
after_estimator = LandPriceEstimator(model=model, blocks=blocks_after)

blocks_before_pred = before_estimator.predict()
blocks_before_pred["land_value_per_100m2"] = blocks_before_pred["land_value"] / blocks_before_pred["site_area"] * 100

blocks_after_pred = after_estimator.predict()
blocks_after_pred["land_value_per_100m2"] = blocks_after_pred["land_value"] / blocks_after_pred["site_area"] * 100

blocks_full_value = transfer_baseline_prices(
    after_blocks=blocks_after_pred,
    before_blocks=blocks_before_pred,
    scenario_mode=False,
)

blocks_full_value = blocks_full_value.replace([np.inf, -np.inf], np.nan)
blocks_full_value = blocks_full_value.fillna(0)

# 3) Только визуализация и статистика по процентным изменениям
scenario_result = plot_scenario_impact(
    blocks=blocks_full_value,
    target_idx=target_idx,          
    target_id_column="id",   # если колонка называется иначе — укажи здесь
    figsize=(25, 35),
)

# **Инвестиционная** привлекактельтность 

In [ ]:
from blocksnet.enums import LandUse

benchmarks_demo = {
    LandUse.RESIDENTIAL: {
        "cost_build": 45_000,
        "price_sale": 120_000,
        "construction_years": 3,
        "sale_years": 3,
        "opex_rate": 800,
    },
    LandUse.BUSINESS: {
        "cost_build": 55_000,
        "rent_annual": 25_000,
        "rent_years": 12,
        "construction_years": 4,
        "opex_rate": 1_300,
    },
    LandUse.RECREATION: {
        "cost_build": 20_000,
        "rent_annual": 4_500,
        "rent_years": 15,
        "construction_years": 3,
        "opex_rate": 1_000,
    },
    LandUse.SPECIAL: {
        "cost_build": 35_000,
        "rent_annual": 11_000,
        "rent_years": 15,
        "construction_years": 3,
        "opex_rate": 1_500,
    },
    LandUse.INDUSTRIAL: {
        "cost_build": 38_000,
        "rent_annual": 14_800,
        "rent_years": 12,
        "construction_years": 3,
        "opex_rate": 700,
    },
    LandUse.AGRICULTURE: {
        "cost_build": 25_000,
        "rent_annual": 6_500,
        "rent_years": 15,
        "construction_years": 3,
        "opex_rate": 300,
    },
    LandUse.TRANSPORT: {
        "cost_build": 18_000,
        "rent_annual": 6_200,
        "rent_years": 15,
        "construction_years": 3,
        "opex_rate": 600,
    },
}

In [ ]:
from urbanomy.methods.investment_potential import prepare_investment_input


potential_df = pd.read_csv("./data/land_use_potentials.csv")

investment_input = prepare_investment_input(
    gdf = blocks_after,
)

investment_input.head()

In [ ]:
from urbanomy.methods.investment_potential import InvestmentAttractivenessAnalyzer

an = InvestmentAttractivenessAnalyzer(benchmarks=benchmarks_demo)
summary = an.calculate_investment_metrics(investment_input, discount_rate=0.18)
summary

In [ ]:
# 1) кладём summary на нужный индекс
summary_idxed = summary.copy()
summary_idxed.index = [target_idx]  # если одна строка; иначе set_index по id

# 2) собираем слой
scn = blocks_after[["geometry"]].join(summary_idxed, how="left")
scn["is_project"] = scn.index == target_idx

column = "EI"
vmin, vmax = 0, 100

fig, ax = plt.subplots(figsize=(15, 10))
scn.loc[~scn["is_project"]].plot(ax=ax, color="lightgrey", edgecolor="0.7", linewidth=0.7)

scn_proj = scn.loc[scn["is_project"]]
if len(scn_proj):
    scn_proj.plot(
        ax=ax,
        column=column,
        cmap="RdYlGn",
        legend=True,
        edgecolor="black",
        linewidth=0.9,
        vmin=vmin,
        vmax=vmax,
    )
    for _, row in scn_proj.dropna(subset=[column]).iterrows():
        x, y = row.geometry.representative_point().coords[0]
        ax.text(x, y, f"{row[column]:.2f}", ha="center", va="center", fontsize=10, color="black")

ax.set_axis_off()
plt.show()


In [ ]:
from urbanomy.methods.socio_economic_indicators.sei_calculate import SEREstimator

deafaut_cfg = {
    "population": 300_000,
}

est = SEREstimator(deafaut_cfg) # or use project_cfg
result = est.compute(scn, pretty=True)
result